# Demand Pulse — Data Load & First Look

Loading 90 days of order data across 7 cities. This notebook confirms 
the schema matches expectations and surfaces any data quality 
surprises before we start analysis.

In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

df = pd.read_csv("../data/case3_food_delivery_orders.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f"Shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"Cities: {df['city'].nunique()} unique")
print(f"Cuisines: {df['cuisine'].nunique()} unique")

Shape: (50000, 8)
Date range: 2025-01-01 00:00:00 → 2025-03-31 23:58:00
Cities: 7 unique
Cuisines: 9 unique


## Schema check

In [3]:
df.dtypes

order_id                     object
timestamp            datetime64[ns]
city                         object
restaurant_id                object
cuisine                      object
order_value                   int64
delivery_time_min             int64
surge_applied                 int64
dtype: object

## Sample rows

In [4]:
df.head(5)

,order_id,timestamp,city,restaurant_id,cuisine,order_value,delivery_time_min,surge_applied
0,ORD138520,2025-01-01 00:00:00,Delhi,R0521,Fast Food,185,31,0
1,ORD132134,2025-01-01 00:06:00,Mumbai,R0021,Italian,792,46,0
2,ORD111938,2025-01-01 00:36:00,Chennai,R0221,Continental,702,34,0
3,ORD134651,2025-01-01 00:51:00,Pune,R0498,Italian,468,35,0
4,ORD143104,2025-01-01 01:17:00,Delhi,R0448,Continental,887,50,0


## Summary statistics

In [5]:
df.describe(include='all')

,order_id,timestamp,city,restaurant_id,cuisine,order_value,delivery_time_min,surge_applied
count,50000,50000,50000,50000,50000,50000.000000,50000.000000,50000.000000
unique,50000,NaN,7,800,9,NaN,NaN,NaN
top,ORD138520,NaN,Bangalore,R0729,South Indian,NaN,NaN,NaN
freq,1,NaN,10776,88,5660,NaN,NaN,NaN
mean,NaN,2025-02-14 23:27:24.829200128,NaN,NaN,NaN,330.913020,40.408600,0.238740
min,NaN,2025-01-01 00:00:00,NaN,NaN,NaN,80.000000,15.000000,0.000000
25%,NaN,2025-01-23 12:54:30,NaN,NaN,NaN,190.000000,31.000000,0.000000
50%,NaN,2025-02-14 17:56:00,NaN,NaN,NaN,288.000000,40.000000,0.000000
75%,NaN,2025-03-09 14:29:45,NaN,NaN,NaN,434.000000,49.000000,0.000000
max,NaN,2025-03-31 23:58:00,NaN,NaN,NaN,1215.000000,100.000000,1.000000


## Null check

In [6]:
nulls = df.isnull().sum()
nulls[nulls > 0] if nulls.sum() > 0 else "No nulls."

'No nulls.'

## Surge baseline
What fraction of orders currently carry surge, by city?

In [7]:
surge_by_city = df.groupby('city')['surge_applied'].agg(['mean', 'sum', 'count'])
surge_by_city['mean'] = (surge_by_city['mean'] * 100).round(1)
surge_by_city.columns = ['surge_pct', 'surge_count', 'total_orders']
surge_by_city.sort_values('total_orders', ascending=False)

,surge_pct,surge_count,total_orders
city,,,
Bangalore,24.1,2595,10776
Mumbai,24.1,2417,10022
Delhi,23.5,1923,8171
Hyderabad,23.9,1551,6493
Pune,23.3,1286,5526
Chennai,24.0,1206,5031
Kolkata,24.1,959,3981


In [8]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour

print('=== Demand by hour (orders) ===')
hourly = df.groupby('hour').size()
print(hourly.sort_index())

print()
print('=== Surge rate by hour ===')
hourly_surge = df.groupby('hour')['surge_applied'].mean() * 100
print(hourly_surge.sort_index().round(1))

print()
print('=== Correlation between demand and surge rate (should be high if surge tracks demand) ===')
print(round(hourly.corr(hourly_surge), 3))

print()
print('=== Top-volume city for forecast: Bangalore ===')
bng = df[df.city == 'Bangalore']
print(f'Bangalore orders: {len(bng)}')
print(f'Date range: {bng.timestamp.min()} → {bng.timestamp.max()}')
print(f'Days: {bng.timestamp.dt.date.nunique()}')

=== Demand by hour (orders) ===
hour
0      312
1      302
2      277
3      306
4      315
5      617
6      957
7     1216
8     1586
9     1825
10    2109
11    2410
12    4571
13    4826
14    2407
15    1871
16    1586
17    1832
18    3683
19    5586
20    5052
21    3877
22    1863
23     614
dtype: int64

=== Surge rate by hour ===
hour
0      5.8
1      5.6
2      7.2
3      4.6
4      5.4
5      4.9
6      4.8
7      7.0
8      5.5
9      5.4
10     5.7
11     5.8
12    28.9
13    29.7
14     6.4
15     6.0
16     6.3
17     6.4
18     5.7
19    52.1
20    53.6
21    52.8
22     5.4
23     5.0
Name: surge_applied, dtype: float64

=== Correlation between demand and surge rate (should be high if surge tracks demand) ===
0.812

=== Top-volume city for forecast: Bangalore ===
Bangalore orders: 10776
Date range: 2025-01-01 02:33:00 → 2025-03-31 23:12:00
Days: 90
